In [1]:


import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")



Project root: c:\Users\jsteve\OneDrive - Burlington\Documents\Python Scripts\Projects\Data-Science-Projects\medicare-claims-intelligence-agent


In [2]:
from database.connection import (
    get_connection,
    register_high_cost_persistence,
    register_member_year,
)

from database.schema import (
    register_analytical_views,
)


con = get_connection()

register_member_year(con)
register_high_cost_persistence(con)
register_analytical_views(con)

print("DuckDB connection established.")

DuckDB connection established.


In [3]:
query = """
SELECT
    COUNT(*) AS member_years,
    COUNT(DISTINCT DESYNPUF_ID) AS unique_beneficiaries
FROM member_year
"""

result = con.execute(query).fetchdf()

result

,member_years,unique_beneficiaries
0,343644,116352


In [4]:
# Create the annual population

query = """
SELECT
    year,
    COUNT(*) AS member_years,
    COUNT(DISTINCT DESYNPUF_ID) AS beneficiaries
FROM member_year
GROUP BY year
ORDER BY year
"""

annual_population = (
    con.execute(query)
    .fetchdf()
)

annual_population

,year,member_years,beneficiaries
0,2008,116352,116352
1,2009,114538,114538
2,2010,112754,112754


In [5]:
# Validating our logic based on orthogonal result

query = """
SELECT
    COUNT(*) AS member_years,

    SUM(
        CASE
            WHEN has_inpatient_utilization
            THEN 1
            ELSE 0
        END
    ) AS inpatient_member_years,

    100.0 * AVG(
        CASE
            WHEN has_inpatient_utilization
            THEN 1
            ELSE 0
        END
    ) AS inpatient_prevalence_pct

FROM member_year
"""

inpatient_summary = con.execute("""
    SELECT
        COUNT(*) AS member_years,
        SUM(CASE WHEN inpatient_claims > 0 THEN 1 ELSE 0 END)
            AS inpatient_member_years,
        100.0 * AVG(CASE WHEN inpatient_claims > 0 THEN 1.0 ELSE 0.0 END)
            AS inpatient_prevalence_pct
    FROM member_year
""").fetchdf()

inpatient_summary

,member_years,inpatient_member_years,inpatient_prevalence_pct
0,343644,46407.0,13.504382


Both our member population and inpatient prevalence match our data from our previous work.  Our DuckDB queries are working correctly.

In [6]:
# Checking functions

from tools.population import get_population_summary


population_summary = get_population_summary()

population_summary

{'member_years': 343644,
 'unique_beneficiaries': 116352,
 'first_year': 2008,
 'last_year': 2010,
 'mean_annual_cost': 4605.670344891807,
 'median_annual_cost': 1970.0,
 'inpatient_prevalence_pct': 13.504382442294933,
 'outpatient_prevalence_pct': 54.249746831022804,
 'carrier_prevalence_pct': 76.4561581171212,
 'rx_prevalence_pct': 71.53740498888385}

In [7]:
query = """
SELECT
    COUNT(*) AS member_years,
    COUNT(h.DESYNPUF_ID) AS persistence_labeled_rows,
    COUNT(*) - COUNT(h.DESYNPUF_ID) AS unlabeled_rows

FROM member_year m

LEFT JOIN high_cost_persistence h
    ON m.DESYNPUF_ID = h.DESYNPUF_ID
"""

con.execute(query).fetchdf()

,member_years,persistence_labeled_rows,unlabeled_rows
0,343644,338262,5382


In [8]:
query = """
SELECT
    COUNT(*) AS member_years,
    AVG(total_annual_cost) AS mean_annual_cost,
    MEDIAN(total_annual_cost) AS median_annual_cost,

    100.0 * AVG(
        CASE
            WHEN has_inpatient_utilization
            THEN 1.0
            ELSE 0.0
        END
    ) AS inpatient_prevalence_pct

FROM analytical_member_year
"""

con.execute(query).fetchdf()

,member_years,mean_annual_cost,median_annual_cost,inpatient_prevalence_pct
0,343644,4605.670345,1970.0,13.504382


In [9]:
# Inspecting high cost persistence file

from pathlib import Path

PERSISTENCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "high_cost_persistence.parquet"
)

con = get_connection()

persistence_schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{PERSISTENCE_PATH.as_posix()}'
    )
    """
).fetchdf()

persistence_schema

,column_name,column_type,null,key,default,extra
0,DESYNPUF_ID,VARCHAR,YES,None,None,None
1,high_cost_years,BIGINT,YES,None,None,None
2,very_high_cost_years,BIGINT,YES,None,None,None
3,mean_cost_percentile,DOUBLE,YES,None,None,None
4,high_cost_pattern,VARCHAR,YES,None,None,None


In [10]:
con.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{PERSISTENCE_PATH.as_posix()}'
    )
    LIMIT 5
    """
).fetchdf()


,DESYNPUF_ID,high_cost_years,very_high_cost_years,mean_cost_percentile,high_cost_pattern
0,00013D2EFD8E45D1,1,0,0.768619,Episodic high cost
1,00016F745862898F,2,2,0.747271,Recurrent high cost
2,0001FDD721E223DC,0,0,0.331560,Never high cost
3,00021CA6FF03E670,0,0,0.134557,Never high cost
4,00024B3D2352D2D0,0,0,0.228845,Never high cost


In [11]:
# Checking Segment tool

from core.models import SegmentName
from tools.segments import get_segment_profile


persistent_profile = get_segment_profile(
    SegmentName.PERSISTENT_COMPLEX
)

persistent_profile

{'segment': 'Persistent high-cost complex',
 'member_years': 3895,
 'unique_beneficiaries': 1453,
 'mean_annual_cost': 29925.566110397947,
 'mean_conditions': 6.486264441591785,
 'inpatient_prevalence_pct': 71.96405648267009,
 'mean_inpatient_days': 9.242105263157894,
 'mean_outpatient_claims': 8.600513478818998,
 'mean_carrier_claims': 47.75121951219512,
 'mean_rx_events': 33.90320924261874,
 'mean_rx_days_supply': 1081.540436456996}

In [12]:
# Testing compare_segments tool

from tools.segments import compare_segments

comparison = compare_segments(
    SegmentName.PERSISTENT_COMPLEX,
    SegmentName.PHARMACY_INTENSIVE,
)

comparison


{'segment_a': {'segment': 'Persistent high-cost complex',
  'member_years': 3895,
  'unique_beneficiaries': 1453,
  'mean_annual_cost': 29925.566110397947,
  'mean_conditions': 6.486264441591785,
  'inpatient_prevalence_pct': 71.96405648267009,
  'mean_inpatient_days': 9.242105263157894,
  'mean_outpatient_claims': 8.600513478818998,
  'mean_carrier_claims': 47.75121951219512,
  'mean_rx_events': 33.90320924261874,
  'mean_rx_days_supply': 1081.540436456996},
 'segment_b': {'segment': 'Pharmacy-intensive chronic',
  'member_years': 10169,
  'unique_beneficiaries': 7552,
  'mean_annual_cost': 7727.750024584521,
  'mean_conditions': 5.385878650801455,
  'inpatient_prevalence_pct': 0.0,
  'mean_inpatient_days': 0.0,
  'mean_outpatient_claims': 5.132559740387452,
  'mean_carrier_claims': 29.395122430917496,
  'mean_rx_events': 60.468679319500446,
  'mean_rx_days_supply': 2046.2552856721409},
 'differences': {'mean_annual_cost': {'segment_a_value': 29925.566110397947,
   'segment_b_value': 

In [13]:
# Test cost tool

from tools.costs import get_cost_concentration


top_10_concentration = (
    get_cost_concentration(0.10)
)

top_10_concentration

{'top_fraction': 0.1,
 'top_percent': 10.0,
 'positive_member_years': 302102,
 'selected_member_years': 30210,
 'total_positive_spending': 1582713170.0,
 'selected_spending': 780230690.0,
 'spending_share_pct': 49.29703655653538}

In [14]:
for fraction in [
    0.01,
    0.05,
    0.10,
    0.20,
]:
    result = get_cost_concentration(
        fraction
    )

    print(
        f"Top {result['top_percent']:.0f}%: "
        f"{result['spending_share_pct']:.2f}% "
        f"of positive spending"
    )

Top 1%: 12.53% of positive spending
Top 5%: 34.67% of positive spending
Top 10%: 49.30% of positive spending
Top 20%: 66.15% of positive spending


In [16]:
# Validating conditions test

from tools.conditions import (
    get_condition_prevalence,
)

persistent_conditions = (
    get_condition_prevalence(
        SegmentName.PERSISTENT_COMPLEX
    )
)



import pandas as pd

pd.DataFrame(
    persistent_conditions["conditions"]
)

,condition,prevalence_pct
0,Diabetes,94.326059
1,Ischemic Heart Disease,94.274711
2,Heart Failure,89.037227
3,Chronic Kidney Disease,85.211810
4,Depression,60.744544
5,COPD,58.382542
6,Alzheimer's / Dementia,57.432606
7,Osteoporosis,33.966624
8,Rheumatoid Arthritis / Osteoarthritis,33.093710
9,Stroke / TIA,21.797176


In [17]:
#LLM initialization smoke test

from core.llm import get_llm

llm = get_llm()

response = llm.invoke(
    "Reply with exactly: Claims agent online"
)

print(response.content)

Claims agent online
